# Notebook 04: Gold atomic Finance fact

## Runtime contract

- **Inputs:** current-state Silver ACDOCA and optional BKPF header rows; Delta-enabled Fabric Spark.
- **Parameters:** `run_id` selects durable `audit.extraction_changes` keys; blank performs the legacy full/manual upsert. A missing target bootstraps from all current Silver rows.
- **Output:** the existing `fct.finance_transaction` schema and deterministic complete ACDOCA key are retained.
- **Incrementality:** ACDOCA changes and all lines under changed/deleted BKPF headers are re-resolved. Only journal-confirmed ACDOCA removals absent from current Silver are deleted.
- **Dimensions:** posting-date SCD2 resolution also respects SAP validity intervals; historical dimension rows remain usable.
- **Replay/failure:** unchanged replays do not rewrite facts; grain assertions run before the atomic fact MERGE and append evidence to `audit.finance_fact_reconciliation`.

- **Master-derived repairs:** journaled master natural keys (including CEPC-derived division dependencies) re-resolve matching fact keys using posting-date versions; unchanged historical mappings are not restated.

- **Additive currency migration:** persisted `RTCUR` identifies TSL. A legacy target lacking it receives a one-time full current-source backfill in the same schema-evolving Delta MERGE; unexplained legacy keys fail before writing. Existing amount names and transaction SKs are unchanged. Delta 3.2+ is required for atomic additive schema evolution.


In [ ]:
run_id = ""  # Blank: full/manual upsert; pipeline supplies the durable run ID.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
from functools import reduce
import datetime
import uuid

if not isinstance(run_id, str):
    raise TypeError("run_id must be a string")
run_id = run_id.strip()
BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
SLV = f"{BRZ_SLV_LH}.slv"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
HIGH_DATE = "9999-12-31"
BATCH_ID = run_id or str(uuid.uuid4())
RUN_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)


In [ ]:
MASTER_SOURCE_KEYS = {
    "t001": ["MANDT", "BUKRS"],
    "ska1": ["MANDT", "KTOPL", "SAKNR"],
    "skat": ["MANDT", "SPRAS", "KTOPL", "SAKNR"],
    "csks": ["MANDT", "KOKRS", "KOSTL", "DATBI"],
    "cepc": ["MANDT", "KOKRS", "PRCTR", "DATBI"],
    "lfa1": ["MANDT", "LIFNR"],
    "kna1": ["MANDT", "KUNNR"],
}
FINANCE_MASTER_MAP = {
    "t001": {"MANDT": "RCLNT", "BUKRS": "RBUKRS"},
    "ska1": {"MANDT": "RCLNT", "SAKNR": "RACCT"},
    "skat": {"MANDT": "RCLNT", "SAKNR": "RACCT"},
    "csks": {"MANDT": "RCLNT", "KOKRS": "KOKRS", "KOSTL": "RCNTR"},
    "cepc": {"MANDT": "RCLNT", "KOKRS": "KOKRS", "PRCTR": "PRCTR"},
    "lfa1": {"MANDT": "RCLNT", "LIFNR": "LIFNR"},
    "kna1": {"MANDT": "RCLNT", "KUNNR": "KUNNR"},
}
PLAN_MASTER_MAP = {
    "t001": {"MANDT": "_CLIENT", "BUKRS": "COMPANY_CODE"},
    "ska1": {"MANDT": "_CLIENT", "SAKNR": "GL_ACCOUNT"},
    "skat": {"MANDT": "_CLIENT", "SAKNR": "GL_ACCOUNT"},
    "csks": {"MANDT": "_CLIENT", "KOKRS": "_KOKRS", "KOSTL": "COST_CENTER"},
    "cepc": {"MANDT": "_CLIENT", "KOKRS": "_KOKRS", "PRCTR": "PROFIT_CENTER"},
}


ACDOCA_KEYS = ["RCLNT", "RLDNR", "RBUKRS", "GJAHR", "BELNR", "DOCLN"]
BKPF_KEYS = ["MANDT", "BUKRS", "GJAHR", "BELNR"]


def key_subset(rows, keys_df, keys: list[str], how: str = "left_semi"):
    left, right = rows.alias("_rows"), keys_df.select(*keys).distinct().alias("_keys")
    match = reduce(lambda a, b: a & b, [
        F.col(f"_rows.{key}").eqNullSafe(F.col(f"_keys.{key}")) for key in keys
    ])
    return left.join(right, match, how)


def read_change_keys(table_name: str, source, keys: list[str], change_type=None):
    journal = f"{AUD}.extraction_changes"
    if not spark.catalog.tableExists(journal):
        raise RuntimeError(f"Incremental Gold requires {journal}; do not infer deletes from batch IDs")
    changes = spark.table(journal).filter(
        (F.col("RUN_ID") == F.lit(run_id))
        & (F.lower(F.col("TABLE_NAME")) == F.lit(table_name))
    )
    invalid_type = ~F.col("CHANGE_TYPE").isin("INSERT", "UPDATE", "DELETE")
    if changes.filter(F.col("CHANGE_TYPE").isNull() | invalid_type).limit(1).count():
        raise ValueError(f"{table_name}: invalid journal CHANGE_TYPE")
    if change_type is not None:
        changes = changes.filter(F.col("CHANGE_TYPE") == F.lit(change_type))
    names = F.json_object_keys(F.col("KEY_JSON"))
    missing_key = reduce(lambda a, b: a | b, [
        ~F.array_contains(names, key) for key in keys
    ])
    if changes.filter(names.isNull() | missing_key).limit(1).count():
        raise ValueError(f"{table_name}: journal KEY_JSON must encode complete keys {keys}")
    key_schema = T.StructType([source.schema[key] for key in keys])
    return (changes.select(F.from_json("KEY_JSON", key_schema, {"mode": "FAILFAST"}).alias("_KEY"))
            .select("_KEY.*").dropDuplicates(keys))


def master_scope_keys(rows, scope_keys: list[str], planning: bool = False):
    """Repair affected natural keys, never replace historical lookups with IS_CURRENT."""
    journal = spark.table(f"{AUD}.extraction_changes").filter(F.col("RUN_ID") == F.lit(run_id))
    mappings = PLAN_MASTER_MAP if planning else FINANCE_MASTER_MAP
    affected = None
    for table_name, mapping in mappings.items():
        if not journal.filter(F.lower(F.col("TABLE_NAME")) == table_name).limit(1).count():
            continue
        source = spark.table(f"{SLV}.{table_name}")
        changed = read_change_keys(table_name, source, MASTER_SOURCE_KEYS[table_name])
        natural_keys = list(mapping)
        changed_keys = changed.select(*natural_keys).distinct()
        if table_name == "cepc":
            # Division is derived from CEPC. A changed division label/boundary
            # can also affect other profit centers sharing its old/new KHINR.
            profit_centers = spark.table(f"{DIM}.profit_center")
            changed_pc = key_subset(profit_centers, changed, MASTER_SOURCE_KEYS["cepc"])
            division_codes = changed_pc.select("KHINR").filter(F.col("KHINR").isNotNull()).distinct()
            related_pc = key_subset(profit_centers, division_codes, ["KHINR"]).select(*natural_keys).distinct()
            changed_keys = changed_keys.unionByName(related_pc).distinct()
        left, right = rows.alias("_mrows"), changed_keys.alias("_mkeys")
        conditions = []
        for master_key, row_key in mapping.items():
            condition = F.col(f"_mrows.{row_key}").eqNullSafe(F.col(f"_mkeys.{master_key}"))
            if planning and row_key in {"_CLIENT", "_KOKRS"}:
                # Budget files have no SAP client/controlling area; retain the
                # existing unambiguous conformance requirement instead of inventing one.
                condition = F.col(f"_mrows.{row_key}").isNull() | condition
            conditions.append(condition)
        matching = left.join(right, reduce(lambda a, b: a & b, conditions), "left_semi").select(*scope_keys).distinct()
        affected = matching if affected is None else affected.unionByName(matching)
    return affected.distinct() if affected is not None else None


def business_key_hash(columns):
    return F.sha2(F.concat_ws("||", *[
        F.coalesce(column.cast("string"), F.lit("__NULL__")) for column in columns
    ]), 256)


def assert_fact_grain(source_count: int, output_count: int, duplicate_count: int, label: str) -> None:
    if duplicate_count or source_count != output_count:
        raise RuntimeError(f"{label} grain violation: source={source_count}, output={output_count}, duplicates={duplicate_count}; check dimension/client/ledger ambiguity")


def merge_fact(out, deleted_ids, target: str, key: str, additive_columns: tuple = ()):
    persisted = [column for column in out.columns if column != "_MATCH_SK"]
    if not spark.catalog.tableExists(target):
        out.select(*persisted).write.format("delta").mode("overwrite").saveAsTable(target)
        return
    target_columns = spark.table(target).columns
    missing = [column for column in persisted if column not in target_columns]
    if set(missing) - set(additive_columns):
        raise RuntimeError(f"{target}: unexpected schema additions {missing}; validate the deployed fact contract")
    match_column = "_MATCH_SK" if "_MATCH_SK" in out.columns else key
    matched_ids = out.select(F.col(match_column).alias(key)).unionByName(out.select(key)).distinct()
    removals = key_subset(deleted_ids.select(key).distinct(), matched_ids, [key], "left_anti")
    if match_column != key:
        removals = removals.withColumn(match_column, F.col(key))
    actions = out.withColumn("_DELETE", F.lit(False)).unionByName(
        removals.withColumn("_DELETE", F.lit(True)), allowMissingColumns=True
    )
    if actions.groupBy(match_column).count().filter(F.col("count") > 1).limit(1).count():
        raise RuntimeError(f"{target}: ambiguous re-key actions; back up and repair duplicate source identities before replay")
    values = {column: f"src.`{column}`" for column in persisted}
    compared = [column for column in persisted if column not in {"GOLD_BATCH_ID", "GOLD_LOAD_TIMESTAMP"}]
    changed = "true" if missing else "NOT (" + " AND ".join(f"tgt.`{column}` <=> src.`{column}`" for column in compared) + ")"
    builder = DeltaTable.forName(spark, target).alias("tgt").merge(
        actions.alias("src"), f"tgt.`{key}` = src.`{match_column}`"
    )
    if missing:
        if not hasattr(builder, "withSchemaEvolution"):
            raise RuntimeError("Additive Gold migration requires Delta 3.2+ merge schema evolution; no fact changes were applied")
        builder = builder.withSchemaEvolution()
    (builder.whenMatchedDelete(condition="src._DELETE")
     .whenMatchedUpdate(condition=f"NOT src._DELETE AND ({changed})", set=values)
     .whenNotMatchedInsert(condition="NOT src._DELETE", values=values).execute())


def dim_join(df, dim_name, alias, conditions, date_col="POSTING_DATE"):
    dimension = spark.table(f"{DIM}.{dim_name}").alias(alias)
    match = reduce(lambda left, right: left & right, conditions)
    date = F.col(date_col)
    match = match & date.between(F.col(f"{alias}.EFFECTIVE_FROM"), F.col(f"{alias}.EFFECTIVE_TO"))
    if dim_name in {"cost_center", "profit_center", "division"}:
        match = match & date.between(F.col(f"{alias}.DATAB"), F.col(f"{alias}.DATBI"))
    return df.join(dimension, match, "left")


def finance_scope(all_ac, bk, target_exists: bool, force_full: bool = False):
    if run_id and target_exists:
        affected = read_change_keys("acdoca", all_ac, ACDOCA_KEYS)
        header_keys = read_change_keys("bkpf", bk, BKPF_KEYS).select(
            F.col("MANDT").alias("RCLNT"), F.col("BUKRS").alias("RBUKRS"), "GJAHR", "BELNR"
        )
        header_lines = key_subset(all_ac, header_keys, ["RCLNT", "RBUKRS", "GJAHR", "BELNR"])
        affected = affected.unionByName(header_lines.select(*ACDOCA_KEYS)).distinct()
        master_keys = master_scope_keys(all_ac, ACDOCA_KEYS)
        if master_keys is not None:
            affected = affected.unionByName(master_keys).distinct()
        ac = all_ac if force_full else key_subset(all_ac, affected, ACDOCA_KEYS)
        deleted = key_subset(
            read_change_keys("acdoca", all_ac, ACDOCA_KEYS, "DELETE"),
            all_ac.select(*ACDOCA_KEYS), ACDOCA_KEYS, "left_anti"
        )
    else:
        ac = all_ac
        deleted = all_ac.select(*ACDOCA_KEYS).limit(0)
    return ac, deleted


def validate_finance_currency_backfill(existing, source, confirmed_deleted):
    """Only complete typed source keys or explicit journal removals explain legacy rows."""
    recovery = "Restore authoritative Silver rows or replay their original deletion journal; otherwise back up and explicitly rebuild the legacy finance fact. No fact changes were applied."
    for rows, keys in [(source, ACDOCA_KEYS), (existing, ["FINANCE_TRANSACTION_SK"])]:
        if rows.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
            raise RuntimeError(f"Ambiguous legacy finance currency backfill. {recovery}")
    expected_sk = business_key_hash([F.col(key) for key in ACDOCA_KEYS])
    if existing.filter(~F.col("FINANCE_TRANSACTION_SK").eqNullSafe(expected_sk)).limit(1).count():
        raise RuntimeError(f"Legacy finance key does not match its complete source identity. {recovery}")
    explained = source.select(*ACDOCA_KEYS).unionByName(confirmed_deleted.select(*ACDOCA_KEYS)).distinct()
    if key_subset(existing, explained, ACDOCA_KEYS, "left_anti").limit(1).count():
        raise RuntimeError(f"Unresolved legacy finance currency backfill. {recovery}")


target = f"{FCT}.finance_transaction"
all_ac = spark.table(f"{SLV}.acdoca")
bk = spark.table(f"{SLV}.bkpf").alias("h")
if "RTCUR" not in all_ac.columns:
    raise RuntimeError("Silver ACDOCA.RTCUR is required for TSL; RWCUR is not a substitute")
target_exists = spark.catalog.tableExists(target)
currency_backfill = target_exists and "RTCUR" not in spark.table(target).columns
ac, deleted = finance_scope(all_ac, bk, target_exists, force_full=currency_backfill)
if currency_backfill:
    validate_finance_currency_backfill(spark.table(target), all_ac, deleted)
ac = ac.alias("a")

header_match = (
    F.col("a.RCLNT").eqNullSafe(F.col("h.MANDT"))
    & F.col("a.RBUKRS").eqNullSafe(F.col("h.BUKRS"))
    & F.col("a.GJAHR").eqNullSafe(F.col("h.GJAHR"))
    & F.col("a.BELNR").eqNullSafe(F.col("h.BELNR"))
)
joined = ac.join(bk, header_match, "left")


def a(name, data_type=None):
    value = F.col(f"a.{name}") if name in ac.columns else F.lit(None)
    return value.cast(data_type) if data_type else (value if name in ac.columns else value.cast("string"))


def h(name, data_type=None):
    value = F.col(f"h.{name}") if name in bk.columns else F.lit(None)
    return value.cast(data_type) if data_type else (value if name in bk.columns else value.cast("string"))


# Without a supplied posting date, carry-forward 000 uses January and special periods use December.
fallback_posting_date = F.make_date(
    a("GJAHR", "int"), F.greatest(F.lit(1), F.least(a("POPER", "int"), F.lit(12))), F.lit(1)
)
fact = joined.withColumn("POSTING_DATE", F.coalesce(a("BUDAT", "date"), h("BUDAT", "date"), fallback_posting_date))
fact = dim_join(fact, "company", "co", [a("RCLNT").eqNullSafe(F.col("co.MANDT")), a("RBUKRS").eqNullSafe(F.col("co.BUKRS"))])
fact = dim_join(fact, "gl_account", "gl", [a("RCLNT").eqNullSafe(F.col("gl.MANDT")), F.col("co.KTOPL").eqNullSafe(F.col("gl.KTOPL")), a("RACCT").eqNullSafe(F.col("gl.SAKNR"))])
fact = dim_join(fact, "cost_center", "cc", [a("RCLNT").eqNullSafe(F.col("cc.MANDT")), a("KOKRS").eqNullSafe(F.col("cc.KOKRS")), a("RCNTR").eqNullSafe(F.col("cc.KOSTL"))])
fact = dim_join(fact, "profit_center", "pc", [a("RCLNT").eqNullSafe(F.col("pc.MANDT")), a("KOKRS").eqNullSafe(F.col("pc.KOKRS")), a("PRCTR").eqNullSafe(F.col("pc.PRCTR"))])
fact = dim_join(fact, "division", "dv", [F.col("pc.KHINR").eqNullSafe(F.col("dv.DIVISION_CODE"))])
fact = dim_join(fact, "vendor", "ve", [a("RCLNT").eqNullSafe(F.col("ve.MANDT")), a("LIFNR").eqNullSafe(F.col("ve.LIFNR"))])
fact = dim_join(fact, "customer", "cu", [a("RCLNT").eqNullSafe(F.col("cu.MANDT")), a("KUNNR").eqNullSafe(F.col("cu.KUNNR"))])

source_columns = [
    "RCLNT", "RLDNR", "RBUKRS", "GJAHR", "POPER", "BELNR", "DOCLN",
    "RACCT", "RCNTR", "PRCTR", "KOKRS", "LIFNR", "KUNNR", "DRCRK",
    "BLART", "AWTYP", "SGTXT", "RHCUR", "RKCUR", "RWCUR",
]
out = fact.select(
    business_key_hash([a(column) for column in ACDOCA_KEYS]).alias("FINANCE_TRANSACTION_SK"),
    F.date_format("POSTING_DATE", "yyyyMMdd").cast("int").alias("POSTING_DATE_SK"),
    ((a("GJAHR", "int") * F.lit(100)) + a("POPER", "int")).cast("int").alias("FISCAL_PERIOD_KEY"),
    F.col("co.DIMENSION_SK").alias("COMPANY_SK"),
    F.col("gl.DIMENSION_SK").alias("GL_ACCOUNT_SK"),
    F.col("cc.DIMENSION_SK").alias("COST_CENTER_SK"),
    F.col("pc.DIMENSION_SK").alias("PROFIT_CENTER_SK"),
    F.col("dv.DIMENSION_SK").alias("DIVISION_SK"),
    F.col("ve.DIMENSION_SK").alias("VENDOR_SK"),
    F.col("cu.DIMENSION_SK").alias("CUSTOMER_SK"),
    *[a(column).alias(column) for column in source_columns],
    h("BKTXT").alias("DOCUMENT_HEADER_TEXT"), h("STBLG").alias("REVERSAL_DOCUMENT"),
    a("HSL", "decimal(23,2)").alias("AMOUNT_LOCAL"),
    a("KSL", "decimal(23,2)").alias("AMOUNT_GROUP"),
    a("TSL", "decimal(23,2)").alias("AMOUNT_TRANSACTION"),
    a("WSL", "decimal(23,2)").alias("AMOUNT_OTHER"),
    a("MSL", "decimal(23,3)").alias("QUANTITY"),
    a("BATCH_RUN_ID").alias("SOURCE_BATCH_RUN_ID"), a("EXTRACT_DATE", "date").alias("EXTRACT_DATE"),
    a("LOAD_DATE", "date").alias("SILVER_LOAD_DATE"), a("SOURCE_SYSTEM").alias("SOURCE_SYSTEM"),
    a("SOURCE_FILE_NAME").alias("SOURCE_FILE_NAME"), F.lit(BATCH_ID).alias("GOLD_BATCH_ID"),
    F.lit(RUN_TIMESTAMP).alias("GOLD_LOAD_TIMESTAMP"),
    a("RTCUR", "string").alias("RTCUR"),
)

# Validate before MERGE, not merely after a potentially multiplied write.
source_count = ac.count()
output_count = out.select("FINANCE_TRANSACTION_SK").distinct().count()
duplicate_count = out.groupBy("FINANCE_TRANSACTION_SK").count().filter(F.col("count") > 1).count()
recon = spark.createDataFrame(
    [(BATCH_ID, source_count, output_count, duplicate_count)],
    "gold_batch_id string, source_line_count long, output_distinct_count long, duplicate_key_count long",
).withColumn("gold_load_timestamp", F.lit(RUN_TIMESTAMP))
recon.write.mode("append").format("delta").saveAsTable(f"{AUD}.finance_fact_reconciliation")
display(recon)
assert_fact_grain(source_count, output_count, duplicate_count, "Finance")
deleted_ids = deleted.select(business_key_hash([F.col(key) for key in ACDOCA_KEYS]).alias("FINANCE_TRANSACTION_SK"))
merge_fact(out, deleted_ids, target, "FINANCE_TRANSACTION_SK", additive_columns=("RTCUR",))
